In [1]:
# %load_ext autoreload
# %autoreload 2
# %cd /my_dir/factowl/
# !pip install -e ./
# %cd factowl/

In [2]:
# !python -m spacy download en_core_web_sm

In [3]:
!nvidia-smi

Thu Mar 19 12:09:19 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        Off | 00000000:41:00.0 Off |                  N/A |
|  0%   28C    P0             114W / 370W |      0MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
import json
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from vllm import LLM, SamplingParams
from factowl.retrieval import Retrieval
from factowl.factscorer import FactowlFactScorer
from factowl.io import save_eval_results, save_predictions, load_json_generations
from datasets import load_dataset
from huggingface_hub import login

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
import json
from factowl.retrieval import Retrieval
# wikipedia_page2passages

class WikipediaPage(object):
  def __init__(self, title=None, content=None):
      self.title = title
      self.content = content

In [6]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def page2contexts(title, page_text, tokenizer, paragraph_max_tokens=512):
    stop_sections = {"References", "See also", "External links", "Further reading", "Notes", "Citations", "Sources"}
    cur_section = ""
    cur_paras = []
    chunk_size = 0
    psgs = []
    for line in page_text.split('\n'):
        line = line.strip()
        tokens = tokenizer(line)["input_ids"]
        if chunk_size + len(tokens) > paragraph_max_tokens or line.startswith("==") and line.endswith("=="):
            if len(cur_paras) > 0 and cur_section.strip() not in stop_sections:
                topic = f"{title}: {cur_section}. " if cur_section != '' else f"{title}. "
                psg = topic + ' '.join(cur_paras)
    
                # passages.append({"title": topic.strip(), "text": text})
                psgs.append(psg)
            if line.startswith("==") and line.endswith("=="):
                cur_section = line.strip().strip('=').strip()
            cur_paras.clear()
            chunk_size = 0
        if line.startswith("==") and line.endswith("=="):
            continue
        cur_paras.append(line)
        chunk_size += len(tokens)
    if len(cur_paras) > 0 and cur_section.strip() not in stop_sections:
        topic = f"{title}: {cur_section}. "
        psg = topic + ' '.join(cur_paras)

        psgs.append(psg)
    psgs = [{"title": title, "text": x} for x in psgs]
    # mn = max(len(d["text"]) for d in psgs)
    return psgs
    

In [7]:
hf_token = 'hf_token'
login(token=hf_token)
os.environ["CUDA_VISIBLE_DEVICES"]="1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

model_name = 'meta-llama/Meta-Llama-3-8B-Instruct'
# model_name = 'meta-llama/Llama-3.3-70B-Instruct'
# model_name = 'meta-llama/Llama-3.1-70B-Instruct'
# model_name="Qwen/Qwen2.5-7B-Instruct"
# model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
# model_name="Qwen/Qwen2.5-32B-Instruct"
mn=os.path.basename(model_name)
cache_dir = './.cache/'
data_dir = '/my_dir/wikipedia_dumps/'


LANG="zh" # zh or en
cnp = 1 # Number of context pages retrieved from Wikipedia API.
nsp = 5 # Number of relevant passages retrieved to support a single atomic fact.
context_type = 'wikipedia_api'
SETUP = "retrieval+llama"

In [8]:
vllm_model = LLM(
    model=model_name,
    gpu_memory_utilization=0.8,
    # tensor_parallel_size=2
    # trust_remote_code=True,
)

INFO 03-19 12:09:38 [utils.py:261] non-default args: {'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'meta-llama/Meta-Llama-3-8B-Instruct'}
INFO 03-19 12:09:39 [model.py:541] Resolved architecture: LlamaForCausalLM
INFO 03-19 12:09:39 [model.py:1561] Using max model len 8192
INFO 03-19 12:09:39 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-19 12:09:39 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=6758) INFO 03-19 12:09:41 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='meta-llama/Meta-Llama-3-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantizat

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=6758) INFO 03-19 12:09:50 [default_loader.py:291] Loading weights took 3.25 seconds
(EngineCore_DP0 pid=6758) INFO 03-19 12:09:51 [gpu_model_runner.py:4130] Model loading took 14.96 GiB memory and 5.200879 seconds
(EngineCore_DP0 pid=6758) INFO 03-19 12:09:57 [backends.py:812] Using cache directory: /root/.cache/vllm/torch_compile_cache/34fd8aca4d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=6758) INFO 03-19 12:09:57 [backends.py:872] Dynamo bytecode transform time: 5.72 s
(EngineCore_DP0 pid=6758) INFO 03-19 12:10:02 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.800 s
(EngineCore_DP0 pid=6758) INFO 03-19 12:10:02 [monitor.py:34] torch.compile takes 7.52 s in total
(EngineCore_DP0 pid=6758) INFO 03-19 12:10:03 [gpu_worker.py:356] Available KV cache memory: 2.72 GiB
(EngineCore_DP0 pid=6758) INFO 03-19 12:10:03 [kv_cache_utils.py:1307] GPU KV cache size: 22,288 tokens
(EngineCore_DP0 pid=675

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 12.06it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.85it/s]


(EngineCore_DP0 pid=6758) INFO 03-19 12:10:10 [gpu_model_runner.py:5063] Graph capturing finished in 7 secs, took 2.07 GiB
(EngineCore_DP0 pid=6758) INFO 03-19 12:10:10 [core.py:272] init engine (profile, create kv cache, warmup model) took 19.75 seconds
INFO 03-19 12:10:12 [llm.py:343] Supported tasks: ['generate']


In [9]:
POP_MAP = {
0: "high popularity",
1: "modest populrity",
2: "low popularity"
}

def aggregate_results(topics, generations, popularities, out):
    topic2decisions = {}
    
    for ad in out["decisions"]:
        t = ad["topic"]
        if topic2decisions.get(t) is None:
            topic2decisions[t] = []
        topic2decisions[t].append(ad)
 
    all_true, low_true, med_true, high_true = 0,0,0,0
    all_total, low_total, med_total, high_total = 0,0,0,0
    for t, p in zip(topics, popularities):
        p = int(p)
        if topic2decisions.get(t) is None:
            continue
        decs = topic2decisions[t]
        
        for dec in decs:
            is_sup = dec["is_supported"]
            all_total += 1
            if p == 0:
                high_total += 1
            elif p == 1:
                med_total += 1
            elif p == 2:
                low_total += 1
            else:
                raise Exception(f"Invalid popularity: {p}")
            if is_sup == True:
                all_true += 1
                if p == 0:
                    high_true += 1
                elif p == 1:
                    med_true += 1
                elif p == 2:
                    low_true += 1
    print(f"Calculated precision:")
    print(f"Total: {all_true / all_total} ({all_true} / {all_total})")
    print(f"Low: {low_true / low_total} ({low_true} / {low_total})")
    print(f"Medium: {med_true / med_total} ({med_true} / {med_total})")
    print(f"High: {high_true / high_total} ({high_true} / {high_total})")
    

In [11]:
# Testing dataset load from HF
entity_data=load_dataset("s-nlp/RiDiC", "disasters")["test"]


In [12]:
# GEN_COLS = ('llama3.1:8b', "qwen2.5:7b", "gpt-5-chat")
# domains = ["rivers", "cars", "bad_weather"]
GEN_COLS = ('llama3.1:8b',)
domains = ["rivers", ]


# domains = ["disasters", ]
# generations_data=load_dataset("s-nlp/RIDIC", "LLM_generations_disasters")
for domain in domains:
   entity_data=load_dataset("s-nlp/RiDiC", domain)["test"]
   generations_data=load_dataset("s-nlp/RIDIC", f"LLM_generations_{domain}")["test"]
   topics = list(generations_data[f"title_{LANG}"])
   popularities = list(generations_data["popularity_part_sector"])
   for gen_col in GEN_COLS: 
      topic2cxt  = {x: page2contexts(title=x, page_text=y, tokenizer=tokenizer) for x, y in zip(entity_data[f"title_{LANG}"],
                                                    entity_data[f"wikipedia_page_{LANG}"]) if x is not None and y is not None}
        
      topics = list(generations_data["title_zh"])
      generations = list(generations_data[gen_col])
      # print(len(topics))
      # print(len(generations))
      assert len(topics) == len(generations)
      new_generations = list(x for x, y in zip(generations, topics) if y is not None and topic2cxt.get(y) is not None)
      new_topics = list(y for x, y in zip(generations, topics) if y is not None and topic2cxt.get(y) is not None)
      topics, generations = new_topics, new_generations
       
      
      print(f"Domain: {domain}")
      print(f"Generation column: {gen_col}")      
      print(f"Evaluation for {len(topics)} generations")
        
      base_dir = "./"
      atomic_facts_cache_dir = f"~/cache-ridic-sep/en-llama-filtered/{domain}-{gen_col}-{SETUP}-eval-{mn}/"
      # raise Exception("")
      fs = FactowlFactScorer(model_name=SETUP,
                   data_dir=data_dir,
                   vllm_model=vllm_model,
                   dump_every_int=100,
                   cache_dir=cache_dir,
                   abstain_detection_type="generic",
                   context_retrieval_type="bm25",
                   use_this_topic2content_only=topic2cxt,
                   lang=LANG,
                   filter_facts=True,
                   debug=False)
      assert len(topics) == len(generations)
      save_p=f"/my_dir/factowl_eval/RiDiC/facts/facts_{domain}-{gen_col}_{SETUP}_eval-p{cnp}-c{nsp}-eval-{mn}.tsv"
        
      out = fs.get_score(topics, generations, save_path=save_p, gamma=10, verbose=True)
                
      aggregate_results(topics, generations, popularities, out)

Token indices sequence length is longer than the specified maximum sequence length for this model (516 > 512). Running this sequence through the model will result in indexing errors
[2026-03-19 12:10:24] INFO factscorer.py:84: FactScore is using context retrieval type: wikipedia_api


Domain: rivers
Generation column: llama3.1:8b
Evaluation for 723 generations


100%|██████████| 723/723 [00:00<00:00, 292936.80it/s]

Starting fact generation


Adding requests:   0%|          | 0/701 [00:00<?, ?it/s]

...

100%|██████████| 15138/15138 [00:03<00:00, 4167.76it/s]


Adding requests:   0%|          | 0/15138 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/15138 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

[2026-03-19 12:25:55] INFO fact_validation.py:247: Finished fact filtration. Summary:
[2026-03-19 12:25:55] INFO fact_validation.py:248: ==================================================
[2026-03-19 12:25:55] INFO fact_validation.py:249: Total rows: 15138
[2026-03-19 12:25:55] INFO fact_validation.py:250: Processed: 15138
[2026-03-19 12:25:55] INFO fact_validation.py:251: Skipped (NaN): 0
[2026-03-19 12:25:55] INFO fact_validation.py:252: GOOD facts: 13162 (86.9%)
[2026-03-19 12:25:55] INFO fact_validation.py:253: BAD: 1976 (13.1%)
[2026-03-19 12:25:55] INFO fact_validation.py:254: Clean output saved to: /my_dir/factowl_eval/RiDiC/facts/facts_rivers-llama3.1:8b_retrieval+llama_eval-p1-c5-eval-Meta-Llama-3-8B-Instruct.tsv


Calculated precision:
Total: 0.2135091669347057 (3319 / 15545)
Low: 0.15369649805447472 (1422 / 9252)
Medium: 0.2636793514974105 (1171 / 4441)
High: 0.39200863930885527 (726 / 1852)
